##Environment setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/shimnet_workshop

!pip uninstall -y torchtune
!pip install -r requirements-colab.txt

# if you haven't downloaded the data yet, uncomment the line below
# !python download_files.py --all --overwrite

import matplotlib.pyplot as plt
import numpy as np
from itertools import islice

### Create config file

In [14]:
from omegaconf import OmegaConf
from pathlib import Path

yaml_config = """
model:
  name: ShimNetWithSCRF
  kwargs:
    rensponse_length: 61
    resnponse_head_dims:
    - 128
training:
- batch_size: 64
  learning_rate: 0.001
  max_iters: 1600
- batch_size: 128
  learning_rate: 0.001
  max_iters: 1600
# - batch_size: 512
#   learning_rate: 0.0005
#   max_iters: 12800000
losses_weights:
  clean: 1.0
  noised: 1.0
  response: 1.0
data:
  response_functions_files:
  # Paste path to your SCRF file here
  # - Can be absolute path
  # - Can be relative to repository root
  - ./deconvolved_data/scrf_workshop.pt
  atom_groups_data_file: data/multiplets_10000_parsed.txt
  response_function_stretch_min: 1.0
  response_function_stretch_max: 1.0
  response_function_noise: 0.0
  multiplicity_j1_min: 0.0
  multiplicity_j1_max: 15
  multiplicity_j2_min: 0.0
  multiplicity_j2_max: 15
  number_of_signals_min: 2
  number_of_signals_max: 5
  thf_min: 4.
  thf_max: 4.
  relative_height_min: 1.
  relative_height_max: 1.
  frq_step: 0.30048
  spectrum_noise_max: 0.005
  spectrum_noise_min: 0
  # spectrum width component range [Hz]:
  spectrum_width_min: 0.2
  spectrum_width_max: 0.3
  # multiplet width component range:
  relative_width_min: 1
  relative_width_max: 1
logging:
  step: 10000
  num_plots: 32
metadata:
  spectrometer_frequency: 600.0
"""

config = OmegaConf.create(yaml_config)

run_dir = Path("./my_model/training_1")
run_dir.mkdir(parents=True, exist_ok=True)

OmegaConf.save(config, run_dir / "config.yaml")


##Run training

In [ ]:
!python train.py ./my_model/training_1

##Display results

In [ ]:

#input_spectrum_name="Asarone_deshimmed"
#input_spectrum_name="Asarone_reduce_volume"
input_spectrum_name="lineshape2_deshimmed"
#input_spectrum_name="Cresol_Red_after_different_sample_cut"
#input_spectrum_name="Cresol_Red_after_different_sample"
#input_spectrum_name="multi_singlets_deshimmed"
#input_spectrum_name = "2etnaft_600MHz_deshimmed"
#input_spectrum_name ="styrene_1mm_600MHz_deshimmed"
input_csv = "./INPUT/"+input_spectrum_name+".csv"

weights_file = './my_model/training_1/model.pt'
config_file  = './my_model/training_1/config.yaml'

output_dir = './OUTPUT/training_1_OUTPUT/'

!python ./predict.py \
    --weights {weights_file} \
    --config {config_file} \
    -o {output_dir} \
    {input_csv}

#####

from display_spectra import run

import os


#input_opti_spectrum_name = "Asarone"
input_opti_spectrum_name = "lineshape"
#input_opti_spectrum_name = "Cresol_Red_after_different_sample"
#input_opti_spectrum_name = "Cresol_Red_after_different_sample_cut"
#input_opti_spectrum_name = "multi_singlets"
#input_opti_spectrum_name = "styrene_600MHz"
#input_opti_spectrum_name = "2etnaft_600MHz"

input_spectrum_file = "./INPUT/"+input_spectrum_name+".csv"
corrected_spectrum_file = "./OUTPUT/training_1_OUTPUT/"+input_spectrum_name+"_processed.csv"
reference_spectrum_file = "./OPTI_INPUT_SPECTRA/"+input_opti_spectrum_name+"_shimmed.csv"  # opcjonalne, zostaw None jeśli nie chcesz

os.environ["INPUT_SPECTRUM_FILE"] = input_spectrum_file
os.environ["CORRECTED_SPECTRUM_FILE"] = corrected_spectrum_file or ""
os.environ["REFERENCE_SPECTRUM_FILE"] = reference_spectrum_file or ""

run(input_spectrum_file, corrected_spectrum_file, reference_spectrum_file)